# Hurricane Tracker -- Interactive
This demonstrates the use of input and display widgets in a Jupyter Notebook

In [ ]:
# Import packages
from pathlib import Path
import pandas as pd
import seaborn as sns
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

# GIS pakcages
from arcgis import GIS
from arcgis.features import FeatureLayer
from arcgis.geometry import Polyline
from arcgis.geometry.filters import intersects

# Initialize ArcGIS GIS connection
gis = GIS('home')

# Disable URL warnings
import urllib3
urllib3.disable_warnings()

#Set paths
raw_folder_path = Path.cwd().parent / 'data' / 'raw'
processed_folder_path = Path.cwd().parent / 'data' / 'processed'

In [ ]:
#Set model layers
ibtracs_NA_points = raw_folder_path /'IBTrACS_NA.shp'
if not ibtracs_NA_points.exists():
    print('ibtracs_NA_points.shp file not found.\nBe sure you have unzipped the file.')
    
usa_counties = 'https://services.arcgis.com/P3ePLMYs2RVChkJx/arcgis/rest/services/USA_Counties_Generalized_Boundaries/FeatureServer/0'

In [ ]:
#Convert storm names to spatial dataframe
sdf_allstorms = pd.DataFrame.spatial.from_featureclass(ibtracs_NA_points)

In [ ]:
#Create a spatial dataframe from the counties
counties_lyr = FeatureLayer(usa_counties,gis=gis)

In [ ]:
#Get list of seasons
all_seasons = sdf_allstorms['SEASON'].unique()

In [ ]:
#Function to get a list of storm names for a given season
the_season = 2005
storm_names = sdf_allstorms.loc[sdf_allstorms['SEASON']==the_season]['NAME'].unique()
storm_names

In [ ]:
#Function to extract counties for a given season and name
the_season = 2005
the_name = 'KATRINA'

def get_intersecting_counties(the_season, the_name):
    #Select the points
    sdf_storm_points = sdf_allstorms.loc[
        (sdf_allstorms['SEASON']==the_season) &
        (sdf_allstorms['NAME']==the_name)
        ]

    #Sort points on ISO_TIME
    sdf_storm_points = sdf_storm_points.sort_values(by='ISO_TIME')

    #Create a storm track line from the points
    coords = [[geom.x, geom.y]  for geom in sdf_storm_points["SHAPE"]]
    track_line = Polyline({
        'paths':[coords],
        'spatialReference':sdf_storm_points.spatial.sr
    })

    #Query the counties feature layer for records intersecting the track line geometry
    intersecting_counties = counties_lyr.query(
        geometry_filter=intersects(track_line)
    ).sdf

    return intersecting_counties


In [ ]:
intersecting_counties['STATE_NAME'].unique()

In [ ]:
track_line

In [ ]:
# Create a dictionary of storm names for each storm season

storm_dict = {}
for storm_season in range(2000,2004):
    storm_names = []
    with arcpy.da.SearchCursor(
        in_table = ibtracs_NA_points,
        field_names = ['NAME'], 
        where_clause=f"SEASON = {storm_season}") as cursor:
        for row in cursor:
            if row[0] not in storm_names and row[0] != 'UNNAMED':
                storm_names.append(row[0])
    storm_dict[storm_season] = storm_names